In [1]:
import os, json, time, glob, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, matthews_corrcoef, confusion_matrix)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

assert tf.config.list_physical_devices("GPU"), "NO GPU — abort"

EPOCHS, BATCH_SIZE, TEST_SIZE = 40, 512, 0.30
LR, MOMENTUM = 0.01, 0.9
OUTPUT_DIR = "/kaggle/working"
RESULTS_PATH = os.path.join(OUTPUT_DIR, "focal_stability_results.json")

hits = glob.glob("/kaggle/input/**/ciciot2023_working_set.parquet", recursive=True)
df = pd.read_parquet(hits[0])
feature_cols = [c for c in df.columns if c != "family"]
X_all = df[feature_cols].to_numpy(dtype=np.float32)
le = LabelEncoder(); y_all = le.fit_transform(df["family"].to_numpy())
CLASS_NAMES = list(le.classes_)
N_FEATURES, N_CLASSES = X_all.shape[1], len(CLASS_NAMES)
print(f"X: {X_all.shape} | {N_CLASSES} classes")

def categorical_focal_loss(class_weights, gamma=2.0):
    w = tf.constant(class_weights, dtype=tf.float32)
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        return tf.reduce_sum(w * tf.pow(1.0 - y_pred, gamma) * ce, axis=-1)
    return loss

def build_model(n_inputs, n_output, loss_fn):
    nb = int(round(n_inputs / 2.0))
    visible = keras.Input(shape=(n_inputs, 1))
    e = layers.Dense(n_inputs)(visible); e = layers.BatchNormalization()(e); e = layers.LeakyReLU()(e)
    bottleneck = layers.Dense(nb)(e)
    d = layers.Dense(n_inputs)(bottleneck); d = layers.BatchNormalization()(d); d = layers.LeakyReLU()(d)
    lstm = layers.LSTM(nb, activation="tanh", return_sequences=True)(visible)
    lstm = layers.Dense(n_inputs)(lstm)
    c = layers.Concatenate()([d, lstm])
    c = layers.Conv1D(filters=nb, kernel_size=2, activation="relu")(c)
    c = layers.Flatten()(c)
    out = layers.Dense(n_output, activation="softmax")(c)
    m = keras.Model(visible, out)
    m.compile(optimizer=keras.optimizers.SGD(learning_rate=LR, momentum=MOMENTUM),
              loss=loss_fn, metrics=["accuracy"])
    return m

def run_focal(protocol, seed):
    t0 = time.time()
    np.random.seed(seed); tf.random.set_seed(seed)
    idx = np.arange(len(X_all))
    idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE,
                                      random_state=seed, stratify=y_all)
    if protocol == "A":
        sc = StandardScaler().fit(X_all); Xs = sc.transform(X_all)
        X_tr, y_tr = Xs[idx_tr], y_all[idx_tr]; X_te, y_te = Xs[idx_te], y_all[idx_te]
    else:
        sc = StandardScaler().fit(X_all[idx_tr])
        X_tr, y_tr = sc.transform(X_all[idx_tr]), y_all[idx_tr]
        X_te, y_te = sc.transform(X_all[idx_te]), y_all[idx_te]

    i_fit, i_val = train_test_split(np.arange(len(y_tr)), test_size=0.10,
                                    random_state=seed, stratify=y_tr)
    X_fit, y_fit = X_tr[i_fit], y_tr[i_fit]
    X_val, y_val = X_tr[i_val], y_tr[i_val]

    counts = np.bincount(y_fit, minlength=N_CLASSES).astype(np.float64)
    counts[counts == 0] = 1.0
    cw = counts.sum() / (N_CLASSES * counts); cw = cw / cw.mean()
    loss_fn = categorical_focal_loss(cw.astype(np.float32))

    rs = lambda a: a.reshape(-1, N_FEATURES, 1).astype(np.float32)
    X_fit, X_val, X_te_r = rs(X_fit), rs(X_val), rs(X_te)
    y_fit_oh = keras.utils.to_categorical(y_fit, N_CLASSES)
    y_val_oh = keras.utils.to_categorical(y_val, N_CLASSES)

    ckpt = f"/kaggle/working/_bf_{protocol}_{seed}.weights.h5"
    model = build_model(N_FEATURES, N_CLASSES, loss_fn)
    hist = model.fit(X_fit, y_fit_oh, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
                     validation_data=(X_val, y_val_oh),
                     callbacks=[keras.callbacks.ModelCheckpoint(
                         ckpt, monitor="val_accuracy", mode="max",
                         save_best_only=True, save_weights_only=True, verbose=0)])

    def ev(m):
        p = m.predict(X_te_r, batch_size=2048, verbose=0).argmax(1)
        cm = confusion_matrix(y_te, p)
        return {"accuracy": float(accuracy_score(y_te, p)),
                "balanced_accuracy": float(balanced_accuracy_score(y_te, p)),
                "macro_f1": float(f1_score(y_te, p, average="macro", zero_division=0)),
                "mcc": float(matthews_corrcoef(y_te, p)),
                "confusion_matrix": cm.tolist(),
                "n_empty_pred_classes": int((cm.sum(0) == 0).sum())}

    fin = ev(model)
    tr_pred = model.predict(X_fit, batch_size=2048, verbose=0).argmax(1)
    model.load_weights(ckpt); res_e = ev(model); os.remove(ckpt)

    lc = [float(v) for v in hist.history["loss"]]
    va = hist.history["val_accuracy"]
    out = {"protocol": protocol, "seed": seed, "strategy": "focal", **fin,
           "train_accuracy": float(accuracy_score(y_fit, tr_pred)),
           "rescued_accuracy": res_e["accuracy"],
           "rescued_balanced_accuracy": res_e["balanced_accuracy"],
           "rescued_mcc": res_e["mcc"],
           "best_val_epoch": int(np.argmax(va)) + 1,
           "final_train_loss": lc[-1],
           "n_loss_increases": int((np.diff(lc) > 0).sum()),
           "tail_std": float(np.std(lc[20:])),
           "loss_curve": lc,
           "val_acc_curve": [float(v) for v in va],
           "wall_sec": round(time.time() - t0, 1)}
    keras.backend.clear_session()
    return out

print("Ready.")

X: (547944, 44) | 8 classes
Ready.


In [2]:
N_SEEDS = 20
PROTOCOLS_TO_RUN = ["B"]

results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else []
done = {(r["protocol"], r["seed"]) for r in results}
grid = [(p, s) for p in PROTOCOLS_TO_RUN for s in range(N_SEEDS)]
print(f"{len(grid)} runs, {len(done)} done\n" + "="*78)

for k, (proto, seed) in enumerate(grid, 1):
    if (proto, seed) in done:
        print(f"[{k}/{len(grid)}] skip"); continue
    print(f"[{k}/{len(grid)}] focal protocol={proto} seed={seed} ...", flush=True)
    try:
        r = run_focal(proto, seed)
        results.append(r)
        json.dump(results, open(RESULTS_PATH, "w"), indent=2)
        print(f"    test={r['accuracy']:.4f}  rescued={r['rescued_accuracy']:.4f} "
              f"(ep {r['best_val_epoch']})  loss_incr={r['n_loss_increases']}  "
              f"tail_std={r['tail_std']:.4f}  ({r['wall_sec']}s)")
    except Exception as e:
        import traceback; print(f"    FAILED: {e}"); traceback.print_exc()

print("\nDone ->", RESULTS_PATH)

20 runs, 0 done
[1/20] focal protocol=B seed=0 ...


I0000 00:00:1786393361.678951      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786393361.682270      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


    test=0.7705  rescued=0.7707 (ep 39)  loss_incr=1  tail_std=0.0006  (294.0s)
[2/20] focal protocol=B seed=1 ...
    test=0.7637  rescued=0.7696 (ep 34)  loss_incr=2  tail_std=0.0006  (290.3s)
[3/20] focal protocol=B seed=2 ...
    test=0.7626  rescued=0.7626 (ep 40)  loss_incr=0  tail_std=0.0006  (279.6s)
[4/20] focal protocol=B seed=3 ...
    test=0.7641  rescued=0.7668 (ep 20)  loss_incr=1  tail_std=0.0007  (277.3s)
[5/20] focal protocol=B seed=4 ...
    test=0.7513  rescued=0.7584 (ep 28)  loss_incr=0  tail_std=0.0006  (279.7s)
[6/20] focal protocol=B seed=5 ...
    test=0.7457  rescued=0.7512 (ep 16)  loss_incr=1  tail_std=0.0006  (275.9s)
[7/20] focal protocol=B seed=6 ...
    test=0.7501  rescued=0.7560 (ep 22)  loss_incr=0  tail_std=0.0006  (266.4s)
[8/20] focal protocol=B seed=7 ...
    test=0.7757  rescued=0.7757 (ep 40)  loss_incr=1  tail_std=0.0005  (262.4s)
[9/20] focal protocol=B seed=8 ...
    test=0.7624  rescued=0.7624 (ep 40)  loss_incr=0  tail_std=0.0006  (266.1s)


In [3]:
res = json.load(open(RESULTS_PATH))
f = np.array([r["accuracy"] for r in res])
s = np.array([r["rescued_accuracy"] for r in res])
print("="*70)
print(f"FOCAL, protocol B, n={len(res)}")
print("="*70)
print(f"  final-epoch : {f.mean():.4f} +/- {f.std(ddof=1):.4f}  range {f.min():.4f}-{f.max():.4f}")
print(f"  rescued     : {s.mean():.4f} +/- {s.std(ddof=1):.4f}")
print(f"  var. reduction: {f.std(ddof=1)/s.std(ddof=1):.1f}x")
print(f"  runs below 0.75: {(f<0.75).sum()}/{len(f)}")
print(f"  mean loss increases: {np.mean([r['n_loss_increases'] for r in res]):.1f}")
print()
print("COMPARE — cross-entropy protocol B (n=20, Day 4):")
print("  final 0.8158 +/- 0.0614 | runs below 0.75: 1 | var.red 6.8x")

FOCAL, protocol B, n=20
  final-epoch : 0.7653 +/- 0.0096  range 0.7457-0.7799
  rescued     : 0.7675 +/- 0.0089
  var. reduction: 1.1x
  runs below 0.75: 1/20
  mean loss increases: 1.0

COMPARE — cross-entropy protocol B (n=20, Day 4):
  final 0.8158 +/- 0.0614 | runs below 0.75: 1 | var.red 6.8x
